# PQL Docs — Chunking Strategy Analysis

Goal: understand the corpus well enough to decide on a chunking strategy for RAG/embedding pipelines.

Sections:
0. Setup
1. General Corpus Overview
2. Document Length Distribution Deep-Dive
3. Structural Element Analysis
4. Content Type Segmentation
5. Chunking Strategy Simulation
6. Summary & Recommendations

## 0. Setup

In [1]:
!pip install tiktoken seaborn --quiet

In [ ]:
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import tiktoken

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = '../pql-agent/data/scrape/pql_docs.jsonl'

records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'Loaded {len(df)} documents')
df.head(3)

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')

df['token_count'] = df['full_content'].apply(lambda t: len(enc.encode(t)) if isinstance(t, str) else 0)
print('Token count added. Sample:')
df[['title', 'word_count', 'token_count']].head(5)

## 1. General Corpus Overview

In [ ]:
print('=== Corpus Summary ===')
print(f"Documents:    {len(df)}")
print(f"Total words:  {df['word_count'].sum():,}")
print(f"Total tokens: {df['token_count'].sum():,}")
print()

percentiles = [10, 25, 50, 75, 90, 95, 99]
pct_df = df[['word_count', 'token_count']].describe(percentiles=[p/100 for p in percentiles])
print(pct_df.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, label in zip(axes, ['word_count', 'token_count'], ['Word Count', 'Token Count']):
    sns.histplot(df[col], bins=40, ax=ax, kde=True)
    ax.set_title(f'Distribution of {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('# Documents')
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Median: {df[col].median():.0f}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Word count vs token count scatter — sanity check
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df['word_count'], df['token_count'], alpha=0.5, s=20)
ax.set_xlabel('Word Count')
ax.set_ylabel('Token Count')
ax.set_title('Word Count vs Token Count (should be ~linear)')

m, b = np.polyfit(df['word_count'], df['token_count'], 1)
x_line = np.linspace(0, df['word_count'].max(), 100)
ax.plot(x_line, m * x_line + b, color='red', label=f'tokens ≈ {m:.2f} × words + {b:.0f}')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Avg tokens/word ratio: {(df["token_count"] / df["word_count"].replace(0, np.nan)).mean():.2f}')

In [ ]:
# Outlier docs
tiny = df[df['token_count'] < 50][['title', 'url', 'token_count']].sort_values('token_count')
giant = df[df['token_count'] > 2000][['title', 'url', 'token_count']].sort_values('token_count', ascending=False)

print(f'=== Tiny docs (< 50 tokens): {len(tiny)} ===')
print(tiny.to_string(index=False))
print()
print(f'=== Giant docs (> 2000 tokens): {len(giant)} ===')
print(giant.to_string(index=False))

## 2. Document Length Distribution Deep-Dive

In [ ]:
# Cumulative distribution with chunk size annotations
chunk_sizes = [256, 512, 1024, 2048]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']

sorted_tokens = np.sort(df['token_count'])
cdf = np.arange(1, len(sorted_tokens) + 1) / len(sorted_tokens)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sorted_tokens, cdf * 100, linewidth=2)

for size, color in zip(chunk_sizes, colors):
    pct_below = (df['token_count'] <= size).mean() * 100
    ax.axvline(size, color=color, linestyle='--', alpha=0.8,
               label=f'{size} tokens → {pct_below:.0f}% docs fit entirely')
    ax.axhline(pct_below, color=color, linestyle=':', alpha=0.4)

ax.set_xlabel('Token Count')
ax.set_ylabel('Cumulative % of Documents')
ax.set_title('CDF of Document Token Counts')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: how many docs fit entirely within common chunk sizes
fit_counts = {f'≤{s}': (df['token_count'] <= s).sum() for s in chunk_sizes}
fit_pcts = {k: v / len(df) * 100 for k, v in fit_counts.items()}

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(fit_counts.keys(), fit_counts.values(), color=colors)
for bar, pct in zip(bars, fit_pcts.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{pct:.0f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Chunk Size (tokens)')
ax.set_ylabel('# Docs that fit entirely')
ax.set_title('Docs fitting entirely within each chunk size')
ax.set_ylim(0, len(df) + 15)
ax.axhline(len(df), color='gray', linestyle=':', label=f'Total docs ({len(df)})')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Structural Element Analysis

In [ ]:
def extract_structure(text):
    if not isinstance(text, str):
        return {}
    lines = text.split('\n')
    return {
        'h1_count': sum(1 for l in lines if re.match(r'^# [^#]', l)),
        'h2_count': sum(1 for l in lines if re.match(r'^## [^#]', l)),
        'h3_count': sum(1 for l in lines if re.match(r'^### [^#]', l)),
        'code_block_count': len(re.findall(r'```', text)) // 2,
        'inline_code_count': len(re.findall(r'`[^`\n]+`', text)),
        'list_item_count': sum(1 for l in lines if re.match(r'^\s*[-*+]\s|^\s*\d+\.\s', l)),
        'table_count': len(set(i for i, l in enumerate(lines) if '|' in l and l.strip().startswith('|'))),
        'total_headers': sum(1 for l in lines if re.match(r'^#{1,3} ', l)),
    }

struct_df = df['full_content'].apply(extract_structure).apply(pd.Series)
df = pd.concat([df, struct_df], axis=1)
print('Structural features extracted:')
df[list(struct_df.columns)].describe().round(1)

In [ ]:
# Histograms for structural elements
struct_cols = ['h1_count', 'h2_count', 'h3_count', 'code_block_count', 'inline_code_count', 'list_item_count', 'table_count']
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for ax, col in zip(axes, struct_cols):
    sns.histplot(df[col], bins=20, ax=ax)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Count per doc')

axes[-1].set_visible(False)
plt.suptitle('Structural Elements per Document', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap: structural features + token_count
corr_cols = struct_cols + ['token_count']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Correlation: Structural Features vs Token Count')
plt.tight_layout()
plt.show()

In [ ]:
# Header count vs token count — viability of header-based splitting
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(df['total_headers'], df['token_count'], alpha=0.5,
                     c=df['code_block_count'], cmap='YlOrRd', s=30)
plt.colorbar(scatter, ax=ax, label='Code block count')
ax.set_xlabel('Total Headers (H1+H2+H3)')
ax.set_ylabel('Token Count')
ax.set_title('Headers vs Token Count\n(color = # code blocks)')
plt.tight_layout()
plt.show()

In [ ]:
# Average tokens between H2 headers (mean section length)
def avg_section_tokens(row):
    text = row['full_content']
    if not isinstance(text, str) or row['h2_count'] == 0:
        return np.nan
    sections = re.split(r'^## ', text, flags=re.MULTILINE)
    token_counts = [len(enc.encode(s)) for s in sections if s.strip()]
    return np.mean(token_counts) if token_counts else np.nan

df['avg_section_tokens'] = df.apply(avg_section_tokens, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
valid = df['avg_section_tokens'].dropna()
sns.histplot(valid, bins=30, ax=ax, kde=True)
for size, color in zip([256, 512, 1024], ['red', 'orange', 'green']):
    ax.axvline(size, color=color, linestyle='--', label=f'{size} tokens')
ax.set_xlabel('Avg tokens per H2 section')
ax.set_ylabel('# Documents')
ax.set_title(f'Avg Section Length (H2-split) — {len(valid)} docs with H2 headers')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Mean avg section: {valid.mean():.0f} tokens | Median: {valid.median():.0f} tokens')

## 4. Content Type Segmentation

In [ ]:
def classify_content(row):
    text = row['full_content']
    if not isinstance(text, str) or not text.strip():
        return 'empty'
    lines = text.split('\n')
    total_lines = len(lines)
    if total_lines == 0:
        return 'empty'
    
    # Count lines inside code blocks
    in_block = False
    code_lines = 0
    for line in lines:
        if line.startswith('```'):
            in_block = not in_block
        elif in_block:
            code_lines += 1
    
    code_ratio = code_lines / total_lines
    if code_ratio > 0.3:
        return 'code-heavy'
    elif code_ratio > 0.1:
        return 'mixed'
    else:
        return 'prose-heavy'

df['content_type'] = df.apply(classify_content, axis=1)
print(df['content_type'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
counts = df['content_type'].value_counts()
axes[0].pie(counts.values, labels=counts.index, autopct='%1.0f%%', startangle=90)
axes[0].set_title('Content Type Distribution')

# Box plots by type
order = ['prose-heavy', 'mixed', 'code-heavy']
order = [o for o in order if o in df['content_type'].unique()]
sns.boxplot(data=df[df['content_type'].isin(order)], x='content_type', y='token_count',
            order=order, ax=axes[1])
axes[1].set_title('Token Count by Content Type')
axes[1].set_xlabel('Content Type')
axes[1].set_ylabel('Token Count')
for size, color in zip([512, 1024], ['orange', 'green']):
    axes[1].axhline(size, color=color, linestyle='--', alpha=0.7, label=f'{size} tokens')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nToken stats by content type:')
print(df.groupby('content_type')['token_count'].describe().round(0))

## 5. Chunking Strategy Simulation

In [ ]:
CHUNK_SIZE = 512
OVERLAP = 50

def fixed_size_chunks(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Returns list of token counts for each chunk."""
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunks.append(end - start)
        start += chunk_size - overlap
    return chunks

df['fixed_chunks'] = df['full_content'].apply(fixed_size_chunks)
df['fixed_n_chunks'] = df['fixed_chunks'].apply(len)

all_fixed_sizes = [size for chunks in df['fixed_chunks'] for size in chunks]
print(f'Fixed-size strategy ({CHUNK_SIZE} tokens, {OVERLAP} overlap):')
print(f'  Total chunks produced: {len(all_fixed_sizes)}')
print(f'  Avg chunk size: {np.mean(all_fixed_sizes):.0f} tokens')
print(f'  Chunk size range: {min(all_fixed_sizes)} – {max(all_fixed_sizes)}')

In [ ]:
def header_chunks(text):
    """Split on ## headers, return list of token counts per section."""
    if not isinstance(text, str) or not text.strip():
        return []
    sections = re.split(r'(?m)^(?=## )', text)
    return [len(enc.encode(s)) for s in sections if s.strip()]

df['header_chunks'] = df['full_content'].apply(header_chunks)
df['header_n_chunks'] = df['header_chunks'].apply(len)

all_header_sizes = [size for chunks in df['header_chunks'] for size in chunks]
print(f'Header-based strategy (split on ##):')
print(f'  Total chunks produced: {len(all_header_sizes)}')
print(f'  Avg chunk size: {np.mean(all_header_sizes):.0f} tokens')
print(f'  Chunk size range: {min(all_header_sizes)} – {max(all_header_sizes)}')
print(f'  Chunks > 1024 tokens: {sum(1 for s in all_header_sizes if s > 1024)} ({sum(1 for s in all_header_sizes if s > 1024)/len(all_header_sizes)*100:.1f}%)')
print(f'  Chunks < 50 tokens:   {sum(1 for s in all_header_sizes if s < 50)} ({sum(1 for s in all_header_sizes if s < 50)/len(all_header_sizes)*100:.1f}%)')

In [ ]:
# Compare chunk size distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_fixed_sizes, bins=30, edgecolor='white')
axes[0].set_title(f'Fixed-size chunks ({CHUNK_SIZE}t / {OVERLAP}t overlap)\n'
                  f'{len(all_fixed_sizes)} total chunks')
axes[0].set_xlabel('Chunk size (tokens)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(all_fixed_sizes), color='red', linestyle='--',
                label=f'Mean: {np.mean(all_fixed_sizes):.0f}')
axes[0].legend()

axes[1].hist(all_header_sizes, bins=40, edgecolor='white', color='salmon')
axes[1].set_title(f'Header-based chunks (split on ##)\n'
                  f'{len(all_header_sizes)} total chunks')
axes[1].set_xlabel('Chunk size (tokens)')
axes[1].set_ylabel('Count')
axes[1].axvline(np.mean(all_header_sizes), color='darkred', linestyle='--',
                label=f'Mean: {np.mean(all_header_sizes):.0f}')
for size, color in zip([512, 1024], ['orange', 'green']):
    axes[1].axvline(size, color=color, linestyle=':', alpha=0.7, label=f'{size}t')
axes[1].legend()

plt.suptitle('Chunking Strategy Comparison', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: chunks per document
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df))
ax.bar(x - 0.3, df['fixed_n_chunks'], width=0.4, label='Fixed-size', alpha=0.7)
ax.bar(x + 0.3, df['header_n_chunks'], width=0.4, label='Header-based', alpha=0.7, color='salmon')
ax.set_xlabel('Document (by position)')
ax.set_ylabel('# Chunks')
ax.set_title('Chunks per Document — Fixed-size vs Header-based')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chunk size variance — lower is more uniform
fixed_std = np.std(all_fixed_sizes)
header_std = np.std(all_header_sizes)
print(f'Fixed-size  — std dev of chunk sizes: {fixed_std:.0f} tokens')
print(f'Header-based — std dev of chunk sizes: {header_std:.0f} tokens')
print()

# Edge cases
print('Header-based oversized chunks (> 1024 tokens):')
oversized = []
for _, row in df.iterrows():
    for size in row['header_chunks']:
        if size > 1024:
            oversized.append({'title': row['title'], 'section_tokens': size})
print(pd.DataFrame(oversized).groupby('title')['section_tokens'].max().sort_values(ascending=False).head(10))

## 6. Summary & Recommendations

*(Fill in after running the notebook — key observations and chosen strategy)*

### Key findings
- ...

### Recommendation
- ...

### Next steps
- ...